# Cluster Ingestion Pipeline

## Overview
This notebook orchestrates the ingestion of Excel files from Azure Blob Storage into Delta tables.
It supports two ingestion modes:
- **Full Copy**: Complete data replacement for snapshot datasets
- **Incremental**: Delta merge operations for append-only or slowly changing dimensions

The notebook uses parallel processing via ThreadPool to execute child notebooks concurrently.

## 1. Imports and Dependencies
Import required Python libraries for datetime handling and parallel processing.

In [0]:
from datetime import datetime
from multiprocessing.pool import ThreadPool

## 2. Widget Parameters Configuration
Define input widgets to accept runtime parameters from Azure Data Factory or manual execution.

| Parameter | Description | Default |
|-----------|-------------|---------|
| Environment | Target environment (dev, test, prod) | Empty |
| IngestionGroup | Logical grouping of datasets to process | Empty |
| Datetime | Execution timestamp (optional, defaults to UTC now) | Empty |

In [0]:
# Create input widgets for external parameter passing (e.g., from Azure Data Factory)
dbutils.widgets.text("Environment", "", "Environment")
dbutils.widgets.text("IngestionGroup", "", "Ingestion Group")
dbutils.widgets.text("Datetime", "", "Datetime")

## 3. Environment Setup and Variable Initialization
Retrieve widget values and set up configuration variables for the ingestion process.

In [0]:
# Retrieve parameter values from widgets
environment = dbutils.widgets.get("Environment")
ingestion_group = dbutils.widgets.get("IngestionGroup")

# Set execution datetime - use provided value or default to current UTC timestamp
# Format: YYYY-MM-DD HH:MM:SS
current_datetime = str(dbutils.widgets.get("Datetime") or datetime.now("UTC").strftime("%Y-%m-%d %H:%M:%S"))

# Define child notebook paths for different ingestion strategies
fullcopy_notebook = 'Delta FullCopy Ingestion'       # Notebook for full refresh operations
incremental_notebook = 'Delta Incremental Ingestion' # Notebook for delta merge operations

# Print configuration for debugging and audit trails
print(environment, ingestion_group)

## 4. Data Pipeline Configuration Loading
Connect to Azure SQL Database via JDBC to fetch ingestion metadata from ExcelConfiguration table.

**Filter Criteria:**
- Records not soft-deleted (`DeletedAt IS NULL`)
- Matching the specified Ingestion Group

**Split Strategy:**
- Full Copy: `IsIncremental = false` (snapshot loads)
- Incremental: `IsIncremental = true` (CDC/merge operations)

In [0]:
# Retrieve JDBC connection string from Azure Key Vault
# Scope: kv-velman-dev, Key: velkpi-sqlconnectionstr-jdbc
url = dbutils.secrets.get(scope="kv-velman-dev", key='velkpi-sqlconnectionstr-jdbc')

# Load configuration data from SQL table with filters
df_DataPipeline = (
    spark.read.jdbc(url, "velkpi.ExcelConfiguration")
    .filter("DeletedAt IS NULL")  # Exclude soft-deleted configurations
    .filter(f"IngestionGroup = '{ingestion_group}'")  # Filter by ingestion group parameter
)

# Split dataframes by ingestion strategy
df_FullCopy = df_DataPipeline.filter("IsIncremental = false")      # Full refresh datasets
df_Incremental = df_DataPipeline.filter("IsIncremental = true")    # Incremental/delta datasets

## 5. Full Copy Ingestion Processing
Execute parallel full copy ingestion for all snapshot datasets.

**Process Flow:**
1. Count records requiring full copy
2. Initialize ThreadPool with size equal to record count (one thread per dataset)
3. Collect driver-side data for iteration
4. Map lambda function to execute child notebook for each dataset
5. Pass configuration parameters (paths, ranges, skip rows, etc.)
6. Terminate pool after completion

In [0]:
# Display full copy configuration for monitoring/debugging
display(df_FullCopy)

In [0]:
# Get count of full copy datasets to determine thread pool size
df_fullcopy_count = df_FullCopy.count()
print("Full copy count: " + str(df_fullcopy_count))

# Only process if there are datasets configured for full copy
if df_fullcopy_count > 0:
    # Initialize ThreadPool for parallel execution (one thread per dataset)
    fullcopy_pool = ThreadPool(df_fullcopy_count)

    # Collect configuration data to driver for iteration
    # Note: collect() brings data into driver memory - suitable for configuration metadata
    data_datapipeline = df_FullCopy.collect()

    # Execute child notebook for each full copy dataset in parallel
    fullcopy_pool.map(
        lambda i: (
            dbutils.notebook.run(
                fullcopy_notebook,  # Target notebook path
                3600,               # Timeout in seconds (1 hour)
                {
                    # Core parameters
                    "Environment": str(environment or ""),                    # Target environment
                    "ExcelPath": (i.BlobStagingPath + "/" + i.FileName),      # Full blob path to Excel file
                    "TableName": str(i.TableName or ""),                      # Target Delta table name
                    "SheetName": str(i.SheetName or ""),                      # Excel worksheet name
                    "DataRange": str(i.DataRange or ""),                      # Cell range (e.g., "A1:Z1000")
                    
                    # Schema and structure parameters
                    "IndexColumns": str(i.IndexColumns or ""),                # Columns to use as index
                    "CustomColumnNames": str(i.CustomColumnNames or ""),      # Custom header mappings
                    "ColumnsToSkip": str(i.ColumnsToSkip or ""),              # Columns to exclude from import
                    "RowsToSkip": str(i.RowsToSkip or ""),                    # Header rows to skip
                    
                    # Data orientation and formatting
                    "IsHorizontal": str(i.IsHorizontal or ""),                # Horizontal vs vertical layout
                    "MergeNHeaders": str(i.MergeNHeaders or ""),              # Number of header rows to merge
                    
                    # Metadata parameters
                    "ReportDate": str(i.ReportDate or ""),                    # Business date for the data
                    "ForecastName": str(i.ForecastName or ""),                # Forecast scenario identifier
                    "Datetime": current_datetime,                             # Execution timestamp for audit
                },
            )
        ),
        data_datapipeline,  # Iterable of configuration rows
    )
    
    # Clean up thread pool resources
    fullcopy_pool.terminate()


## 6. Incremental Ingestion Processing
Execute parallel incremental ingestion with watermark-based change detection.

**Additional Parameters for Incremental Loads:**
- `IdColumns`: Primary key columns for merge matching
- `WatermarkColumn`: Column used for change tracking (timestamp/version)
- `WatermarkLookBack`: Historical lookback period for catching late arrivals
- `WatermarkVal`: Last processed watermark value (stored in config)

In [0]:
# Display incremental configuration for monitoring/debugging
display(df_Incremental)

In [0]:
# Get count of incremental datasets to determine thread pool size
df_incremental_count = df_Incremental.count()
print(f"Incremental count: {df_incremental_count}")

# Only process if there are datasets configured for incremental load
if df_incremental_count > 0:
    # Initialize ThreadPool for parallel execution
    incremental_pool = ThreadPool(df_incremental_count)
    
    # Collect configuration data to driver
    data_datapipeline = df_Incremental.collect()
    
    # Execute child notebook for each incremental dataset in parallel
    incremental_pool.map(
        lambda i: (
            dbutils.notebook.run(
                incremental_notebook,  # Target notebook for delta merge operations
                3600,                  # Timeout in seconds (1 hour)
                {
                    # Core identification and environment
                    "Id": str(i.Id or ""),                                    # Configuration record ID
                    "Environment": str(environment or ""),                    # Target environment
                    
                    # Source data location
                    "ExcelPath": (i.BlobStagingPath + "/" + i.FileName),      # Full blob path
                    
                    # Target schema definition
                    "TableName": str(i.TableName or ""),                      # Target Delta table
                    "SheetName": str(i.SheetName or ""),                      # Excel worksheet
                    "DataRange": str(i.DataRange or ""),                      # Cell range specification
                    
                    # Column configuration
                    "IndexColumns": str(i.IndexColumns or ""),                # Business key columns
                    "CustomColumnNames": str(i.CustomColumnNames or ""),      # Header overrides
                    "ColumnsToSkip": str(i.ColumnsToSkip or ""),              # Excluded columns
                    "RowsToSkip": str(i.RowsToSkip or ""),                    # Rows to ignore at top
                    
                    # Layout and formatting
                    "IsHorizontal": str(i.IsHorizontal or ""),                # Data orientation flag
                    "MergeNHeaders": str(i.MergeNHeaders or ""),              # Multi-row header handling
                    
                    # Business metadata
                    "ReportDate": str(i.ReportDate or ""),                    # Reporting period date
                    "ForecastName": str(i.ForecastName or ""),                # Forecast scenario
                    "Datetime": current_datetime,                             # Execution timestamp
                    
                    # Incremental-specific parameters for CDC/SCD logic
                    "IdColumns": str(i.IdColumns or ""),                      # Merge key columns (PK)
                    "WatermarkColumn": str(i.WatermarkColumn or ""),          # Change tracking column
                    "WatermarkLookBack": str(i.WatermarkLookBack or ""),      # Lookback window (days)
                    "WatermarkVal": str(i.WatermarkVal or ""),                # Last processed value
                },
            )
        ),
        data_datapipeline,
    )
    
    # Clean up thread pool resources to prevent memory leaks
    incremental_pool.terminate()